# ✋ Hand Tracking with OpenCV + MediaPipe
**Stack:** OpenCV + MediaPipe  
**Features:**
- 21-landmark detection per hand (up to 2 hands)
- Finger counting (0–5)
- Gesture classification: open hand, closed fist, thumbs up, peace sign, pointing
- Annotated frame output as base64 JPEG

**Note:** Webcam streaming requires running locally. This notebook demonstrates static image inference.

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q opencv-python-headless mediapipe matplotlib pillow

In [ ]:
# Download a sample hand image or use an uploaded one
import urllib.request
import os

os.makedirs('data', exist_ok=True)

# Use a public-domain hand image
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg'

# Better: upload your own hand photo in Colab with:
# from google.colab import files
# uploaded = files.upload()  # upload a hand photo
# image_path = list(uploaded.keys())[0]

# For demo — create a synthetic hand-like image using numpy
import numpy as np
import cv2

# Draw a simple synthetic hand shape
canvas = np.ones((400, 300, 3), dtype=np.uint8) * 200
# Palm
cv2.ellipse(canvas, (150, 250), (80, 100), 0, 0, 360, (210, 180, 140), -1)
# Fingers
for i, (x, y) in enumerate([(90, 150), (120, 110), (155, 95), (190, 105), (230, 130)]):
    cv2.ellipse(canvas, (x, y), (18, 45), -10 + i*5, 0, 360, (210, 180, 140), -1)
# Thumb
cv2.ellipse(canvas, (65, 200), (25, 50), -40, 0, 360, (210, 180, 140), -1)

image_path = 'data/test_hand.jpg'
cv2.imwrite(image_path, canvas)
print(f'Synthetic hand image saved: {image_path}')

import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('Test input image'); plt.axis('off'); plt.show()

In [ ]:
# Run hand tracking inference
import sys; sys.path.insert(0, '.')
from modules.computer_vision.hand_tracking.model import analyze_image
import matplotlib.pyplot as plt
import cv2, base64, numpy as np

with open(image_path, 'rb') as f:
    image_bytes = f.read()

result = analyze_image(image_bytes)

print(f'Hands detected: {result["hands_detected"]}')
print(f'Inference time: {result["inference_ms"]:.1f}ms')

for i, hand in enumerate(result['hands']):
    print(f'\nHand {i+1}:')
    print(f'  Fingers raised: {hand["finger_count"]}')
    print(f'  Gesture:        {hand["gesture"]}')
    print(f'  Landmark count: {len(hand["landmarks"])}')
    print(f'  Wrist (lm 0):   x={hand["landmarks"][0]["x"]:.3f}, y={hand["landmarks"][0]["y"]:.3f}')

In [ ]:
# Show annotated output
annotated = np.frombuffer(result['annotated_image_bytes'], dtype=np.uint8)
annotated = cv2.imdecode(annotated, cv2.IMREAD_COLOR)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)); ax1.set_title('Input'); ax1.axis('off')
ax2.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); ax2.set_title('Annotated (MediaPipe landmarks)'); ax2.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# MediaPipe landmark map
import matplotlib.pyplot as plt
import numpy as np

landmark_names = [
    'WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
    'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
    'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
    'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
    'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP'
]

fig, ax = plt.subplots(figsize=(6, 8))
# Schematic positions
xs = [0.5,0.2,0.15,0.1,0.05, 0.35,0.3,0.28,0.26, 0.5,0.48,0.47,0.46, 0.65,0.67,0.68,0.69, 0.8,0.83,0.85,0.87]
ys = [0.1,0.25,0.45,0.62,0.75, 0.3,0.55,0.72,0.88, 0.25,0.52,0.70,0.87, 0.28,0.54,0.71,0.87, 0.35,0.55,0.68,0.80]

for i, (x, y, name) in enumerate(zip(xs, ys, landmark_names)):
    ax.scatter(x, y, s=100, zorder=5, color='steelblue')
    ax.annotate(f'{i}:{name}', (x, y), fontsize=6, ha='left', va='bottom')

ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
ax.set_title('MediaPipe 21 Hand Landmarks', fontsize=12)
ax.axis('off'); plt.tight_layout(); plt.show()